In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

**<font size="6" color="red">ch01. 허깅페이스 모델 사용</form>**
- Inference API 이용 : 모델의 결과를 server에서
- pipeline() 이용 : 모델을 다운로드 받아 모델의 결과를 local에서
```
허깅페이스 transformer에서 지원하는 task
text-classification : (별칭 sentiment-analysis)	감정 분석, 뉴스 분류, 리뷰 분류 등 문장 분류
zero-shot-classification	: 레이블에 대한 별도 학습 없이 후보 레이블 중에서 분류
text-generation	: GPT 계열 모델을 이용한 텍스트 생성
fill-mask	: 문장 안의 빈칸(마스크)에 들어갈 단어 예측
ner (token-classification의 별칭)	: 개체명 인식(사람, 조직, 장소 등 라벨링)
question-answering	: 주어진 지문(context)을 근거로 질문에 답변
summarization	: 긴 문서를 짧게 요약
translation	: 서로 다른 언어 간 번역
image-to-text	: 이미지 내용을 설명하는 문장 생성
image-classification	: 이미지가 어떤 대상인지 분류

```

In [1]:
import warnings
import os
import logging
 
# 경고 메시지 제거
warnings.filterwarnings('ignore')
 
# transformers 라이브러리의 로깅 레벨을 ERROR로 조정 (경고 숨김)
logging.getLogger("transformers").setLevel(logging.ERROR)
 
# Hugging Face 캐시 관련 symlink 경고 제거
# os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


# 1. 텍스트 기반 감정분석(긍정 / 부정)
- 토큰화 -> 워드임베딩 -> 모델 -> predict : pipeline()함수는 이 단계를 내부적으로 해 줌

In [3]:
from transformers import pipeline
classifier = pipeline(task="text-classification",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
classifier("I've been waiting for a Hugging face course my whole life.")

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982948899269104}]

In [3]:
# 특정 모델의 파라미터와 용량
from transformers import AutoModel
model = AutoModel.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
# 전체 파라미터 수
total_params = sum(p.numel() for p in model.parameters())

In [4]:
from transformers import pipeline
classifier = pipeline(task="sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
# 감정분석할 내용이 많으면 list
classifier([
    "I've been waiting for a Hugging face course my whole life.",
    "I hate this so much!"
])

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982948899269104},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [10]:
classifier("이 영화 정말 최도였어요. 감동적이고 연기도 대단해요!")

[{'label': 'POSITIVE', 'score': 0.975276529788971}]

In [13]:
classifier(["I like you", "I hate you", "힘들어요"])

[{'label': 'POSITIVE', 'score': 0.9998695850372314},
 {'label': 'NEGATIVE', 'score': 0.9991129040718079},
 {'label': 'POSITIVE', 'score': 0.8669533729553223}]

In [14]:
classifier = pipeline(task="sentiment-analysis",
                      model="daekeun-ml/koelectra-small-v3-nsmc")
texts = ['힘들어요', "최고", "싫어요", "너가 좋아"]
classifier(texts)

Device set to use cpu


[{'label': '0', 'score': 0.9957089424133301},
 {'label': '1', 'score': 0.9943759441375732},
 {'label': '0', 'score': 0.9989749193191528},
 {'label': '1', 'score': 0.5141964554786682}]

In [15]:
for text, result in zip(texts, classifier(texts)):
    label = "긍정" if result['label']=='1' else "부정"
    print(f"'{text}' -> {label} {result['score']:.2%}")

'힘들어요' -> 부정 99.57%
'최고' -> 긍정 99.44%
'싫어요' -> 부정 99.90%
'너가 좋아' -> 긍정 51.42%


# 2. 제로샷(Zero-shot-분류)
- 비지도학습

In [ ]:
classifier = pipeline(task='zero-shot-classification',
                      model='facebook/bart-large-,mnli')
classifier("I have a probloem with my iphone that needs to be resolved asap!!",
        candidate_labels=['phone', 'urgent', 'tablet', 'computer'])

In [ ]:
classifier("This is acourse about the Tranformers library.",
         candidate_labels=["education", "business", "phone"])